In [1]:
import torch, transformers
from pathlib import Path
from transformers import BertTokenizer, BertForSequenceClassification

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FINBERT_DIR = PROJECT_ROOT / "models" / "finbert"
print("files:", sorted(p.name for p in FINBERT_DIR.iterdir()))

tokenizer = BertTokenizer.from_pretrained(FINBERT_DIR)
finbert = BertForSequenceClassification.from_pretrained(FINBERT_DIR).eval()

enc = tokenizer("Operating profit fell due to impairment charges .", return_tensors="pt")
with torch.no_grad():
    probs = finbert(**enc).logits.softmax(dim=-1)[0]
print(dict(zip(["negative", "neutral", "positive"], probs.numpy().round(3))))

h:\Desktop\financial-news-intelligence\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


files: ['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin']


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 427.79it/s]


{'negative': np.float32(0.998), 'neutral': np.float32(0.001), 'positive': np.float32(0.001)}


In [5]:
import time, joblib, numpy as np, pandas as pd, keras
from sklearn.metrics import f1_score

LABELS = ["negative", "neutral", "positive"]
train_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "train.csv")
test_df  = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "test.csv")
texts, y_true = test_df["sentence_clean"].tolist(), test_df["label"].tolist()

# Rebuild the RNN word list exactly like Phase 5
vectorizer = keras.layers.TextVectorization(max_tokens=10000, output_sequence_length=50,
                                            standardize="lower", split="whitespace")
vectorizer.adapt(train_df["sentence_clean"].tolist())

tfidf = joblib.load(PROJECT_ROOT / "models" / "tfidf_vectorizer.joblib")
lr    = joblib.load(PROJECT_ROOT / "models" / "tfidf_lr.joblib")
keras_models = {n: keras.models.load_model(PROJECT_ROOT / "models" / f"{n}.keras")
                for n in ["simple_rnn", "bilstm", "bigru"]}

def predict_one(name, text):
    if name == "tfidf_lr":
        return lr.predict(tfidf.transform([text]))[0]
    if name == "finbert":
        enc = tokenizer(text, truncation=True, max_length=128, return_tensors="pt")
        with torch.no_grad():
            return LABELS[finbert(**enc).logits.argmax(dim=-1).item()]
    probs = keras_models[name](vectorizer([text]), training=False)[0]
    return LABELS[int(np.argmax(probs))]

params = {"tfidf_lr": lr.coef_.size + lr.intercept_.size,
          **{n: m.count_params() for n, m in keras_models.items()},
          "finbert": sum(p.numel() for p in finbert.parameters())}

rows = []
for name in ["tfidf_lr", "simple_rnn", "bilstm", "bigru", "finbert"]:
    for t in texts[:10]:
        predict_one(name, t)                       # warm-up (first calls are always slow)
    times, preds = [], []
    for t in texts:
        t0 = time.perf_counter()
        preds.append(predict_one(name, t))
        times.append((time.perf_counter() - t0) * 1000)
    rows.append({"model": name,
                 "test_macro_f1": round(f1_score(y_true, preds, average="macro"), 4),
                 "params": params[name],
                 "median_ms": round(float(np.median(times)), 2),
                 "p95_ms": round(float(np.percentile(times, 95)), 2)})
    print(rows[-1])

table = pd.DataFrame(rows)
table.to_csv(PROJECT_ROOT / "results" / "comparison.csv", index=False)
table

{'model': 'tfidf_lr', 'test_macro_f1': 0.8059, 'params': 19887, 'median_ms': 0.51, 'p95_ms': 0.67}
{'model': 'simple_rnn', 'test_macro_f1': 0.5776, 'params': 501123, 'median_ms': 93.98, 'p95_ms': 119.3}
{'model': 'bilstm', 'test_macro_f1': 0.7448, 'params': 559107, 'median_ms': 261.42, 'p95_ms': 329.67}
{'model': 'bigru', 'test_macro_f1': 0.7414, 'params': 542979, 'median_ms': 332.27, 'p95_ms': 375.4}
{'model': 'finbert', 'test_macro_f1': 0.9249, 'params': 109754115, 'median_ms': 93.5, 'p95_ms': 136.42}


,model,test_macro_f1,params,median_ms,p95_ms
0,tfidf_lr,0.8059,19887,0.51,0.67
1,simple_rnn,0.5776,501123,93.98,119.30
2,bilstm,0.7448,559107,261.42,329.67
3,bigru,0.7414,542979,332.27,375.40
4,finbert,0.9249,109754115,93.50,136.42


In [6]:
import tensorflow as tf

rows_fast = []
for name, m in keras_models.items():
    fast = tf.function(lambda x, m=m: m(x, training=False))       # compile once
    for t in texts[:10]:
        fast(vectorizer([t]))                                     # warm-up (compiles here)
    times = []
    for t in texts[:200]:
        t0 = time.perf_counter()
        fast(vectorizer([t])).numpy()
        times.append((time.perf_counter() - t0) * 1000)
    rows_fast.append({"model": name, "median_ms_compiled": round(float(np.median(times)), 2)})
    print(rows_fast[-1])

{'model': 'simple_rnn', 'median_ms_compiled': 5.03}
{'model': 'bilstm', 'median_ms_compiled': 5.81}
{'model': 'bigru', 'median_ms_compiled': 5.91}


In [7]:
table = table.merge(pd.DataFrame(rows_fast), on="model", how="left")
table.to_csv(PROJECT_ROOT / "results" / "comparison.csv", index=False)
table

,model,test_macro_f1,params,median_ms,p95_ms,median_ms_compiled
0,tfidf_lr,0.8059,19887,0.51,0.67,NaN
1,simple_rnn,0.5776,501123,93.98,119.30,5.03
2,bilstm,0.7448,559107,261.42,329.67,5.81
3,bigru,0.7414,542979,332.27,375.40,5.91
4,finbert,0.9249,109754115,93.50,136.42,NaN


In [9]:
import re

# 1. Every model's prediction for all 492 test sentences (fast batch mode)
X_ids = vectorizer(texts)
preds = {"tfidf_lr": list(lr.predict(tfidf.transform(texts)))}
for n, m in keras_models.items():
    preds[n] = [LABELS[i] for i in m.predict(X_ids, verbose=0).argmax(axis=1)]
fb = []
for i in range(0, len(texts), 32):
    enc = tokenizer(texts[i:i+32], truncation=True, max_length=128, padding=True, return_tensors="pt")
    with torch.no_grad():
        fb += [LABELS[j] for j in finbert(**enc).logits.argmax(dim=-1).tolist()]
preds["finbert"] = fb

err = pd.DataFrame({"sentence": texts, "true": y_true, **preds})

# 2. Label each sentence by type
UP   = r"\b(rose|increased|grew|improved|up|higher|gain|gains)\b"
DOWN = r"\b(fell|decreased|declined|down|lower|loss|losses|dropped)\b"
low = err["sentence"].str.lower()
err["negation"]   = low.str.contains(r"\b(?:not|no|never|without|nor)\b|n't", regex=True)
err["comparison"] = low.str.contains(r"\b(?:from|compared|than)\b", regex=True)
err["number"]     = low.str.contains(r"\d", regex=True)
err["mixed_cues"] = low.str.contains(UP, regex=True) & low.str.contains(DOWN, regex=True)
err["long"]       = low.str.split().str.len() > 30
err.to_csv(PROJECT_ROOT / "results" / "test_predictions.csv", index=False)

# 3. Accuracy of each model on each sentence type
MODELS = ["tfidf_lr", "simple_rnn", "bilstm", "bigru", "finbert"]
TYPES = ["negation", "comparison", "number", "mixed_cues", "long"]
rows = []
for t in ["ALL"] + TYPES:
    sub = err if t == "ALL" else err[err[t]]
    rows.append({"type": t, "n": len(sub), **{m: round((sub[m] == sub["true"]).mean(), 3) for m in MODELS}})
by_type = pd.DataFrame(rows)
by_type.to_csv(PROJECT_ROOT / "results" / "accuracy_by_type.csv", index=False)
by_type

C:\Users\sharm\AppData\Local\Temp\ipykernel_23416\2960970174.py:24: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  err["mixed_cues"] = low.str.contains(UP, regex=True) & low.str.contains(DOWN, regex=True)


,type,n,tfidf_lr,simple_rnn,bilstm,bigru,finbert
0,ALL,492,0.850,0.726,0.803,0.825,0.943
1,negation,18,0.944,1.000,1.000,1.000,0.889
2,comparison,100,0.880,0.580,0.760,0.800,0.920
3,number,271,0.867,0.675,0.790,0.819,0.963
4,mixed_cues,3,0.667,0.333,1.000,0.667,1.000
5,long,94,0.872,0.702,0.809,0.798,0.968


In [10]:
pd.set_option("display.max_colwidth", 200)
fb_wrong = err[err["finbert"] != err["true"]]
print("FinBERT errors:", len(fb_wrong))
for _, r in fb_wrong.iterrows():
    others = " | ".join(f"{m}={r[m][:3]}" for m in ["tfidf_lr", "bilstm", "bigru"])
    print(f"TRUE={r['true'][:3]}  FINBERT={r['finbert'][:3]}  ({others})\n   {r['sentence']}\n")

FinBERT errors: 28
TRUE=pos  FINBERT=neu  (tfidf_lr=neu | bilstm=neu | bigru=neu)
   According to Deputy MD Pekka Silvennoinen the aim is double turnover over the next three years .

TRUE=neu  FINBERT=pos  (tfidf_lr=neu | bilstm=pos | bigru=pos)
   `` Management decided at the end of 2005 to increase cathode copper capacity .

TRUE=neu  FINBERT=pos  (tfidf_lr=pos | bilstm=pos | bigru=neu)
   The original contract was signed last summer .

TRUE=neu  FINBERT=pos  (tfidf_lr=neu | bilstm=neu | bigru=neu)
   The reasons behind the estimate include the rise in 2008 rent levels and several fully-leased office and retail properties , which were completed and added to the company 's investment property portfolio .

TRUE=pos  FINBERT=neu  (tfidf_lr=neu | bilstm=neu | bigru=neu)
   All of Raisio 's divisions recorded an operating profit .

TRUE=pos  FINBERT=neu  (tfidf_lr=neu | bilstm=neu | bigru=neu)
   The company also said that in Poland a profitability program has been launched at the Obornik

In [11]:
cols = ["true", "finbert", "tfidf_lr", "bilstm", "bigru", "sentence"]
fb_wrong[cols].to_csv(PROJECT_ROOT / "results" / "finbert_errors.csv", index=False)

In [12]:
probes = [
    ("Operating profit rose to EUR 13 million .",                          "positive"),
    ("Operating profit did not rise .",                                    "negative"),
    ("Sales increased by 10 % .",                                          "positive"),
    ("Sales did not increase .",                                           "negative"),
    ("Deliveries will be interrupted .",                                   "negative"),
    ("Deliveries will not be interrupted .",                               "neutral"),
    ("The operating loss narrowed to EUR 1 million from EUR 5 million .",  "positive"),
    ("The operating loss widened to EUR 5 million from EUR 1 million .",   "negative"),
    ("Net debt decreased significantly .",                                 "positive"),
    ("Net sales decreased significantly .",                                "negative"),
]
p_texts = [p[0] for p in probes]
out = pd.DataFrame({"sentence": p_texts, "expected": [p[1] for p in probes]})
out["tfidf_lr"] = lr.predict(tfidf.transform(p_texts))
out["bilstm"] = [LABELS[i] for i in keras_models["bilstm"].predict(vectorizer(p_texts), verbose=0).argmax(axis=1)]
enc = tokenizer(p_texts, padding=True, return_tensors="pt")
with torch.no_grad():
    out["finbert"] = [LABELS[j] for j in finbert(**enc).logits.argmax(dim=-1).tolist()]
for m in ["tfidf_lr", "bilstm", "finbert"]:
    print(m, "correct:", (out[m] == out["expected"]).sum(), "/ 10")
out

tfidf_lr correct: 6 / 10
bilstm correct: 6 / 10
finbert correct: 5 / 10


,sentence,expected,tfidf_lr,bilstm,finbert
0,Operating profit rose to EUR 13 million .,positive,positive,positive,positive
1,Operating profit did not rise .,negative,neutral,neutral,positive
2,Sales increased by 10 % .,positive,positive,positive,positive
3,Sales did not increase .,negative,neutral,neutral,positive
4,Deliveries will be interrupted .,negative,neutral,neutral,negative
5,Deliveries will not be interrupted .,neutral,neutral,neutral,negative
6,The operating loss narrowed to EUR 1 million from EUR 5 million .,positive,positive,positive,negative
7,The operating loss widened to EUR 5 million from EUR 1 million .,negative,negative,negative,negative
8,Net debt decreased significantly .,positive,negative,negative,negative
9,Net sales decreased significantly .,negative,negative,negative,negative
